# 01. 미 국채 커브 읽기

par yield로 UST 커브를 만들고, 커브 모양과 포지션의 key rate DV01을 읽는다.

- 데이터: `data/market/*.txt` (`날짜 키 값`, `market-data` 레포가 만든다). 없으면 합성 샘플로 돌아가며, 그때 숫자는 해석하지 않는다.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from quanttool.config import load_config
from quanttool.curves import curve_shape, curve_table, helper_repricing
from quanttool.market import build_market
from quanttool.portfolio import Portfolio
from quanttool.quotes import read_quotes
from quanttool.risk import key_rate

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
files = sorted((ROOT / "data" / "market").glob("*.txt"))
SAMPLE = not files
if SAMPLE:
    files = [ROOT / "tests" / "fixtures" / "market_sample.txt"]
    print("경고: 실제 데이터가 없어 합성 샘플을 쓴다. 아래 숫자는 시장 해석용이 아니다.")
cfg = load_config(ROOT / "configs")
quotes = read_quotes(files)
ASOF = list(quotes)[-1]
market = build_market(ASOF, quotes[ASOF], cfg)
ASOF

## 1. 구축 진단
헬퍼 재가격 오차가 0.01bp를 넘으면 커브를 믿지 않는다.

In [ ]:
helper_repricing(market.built["UST"], ASOF).round({"market": 5, "implied": 5, "error_bp": 6})

## 2. 커브 표와 모양
- `par`: 커브로 다시 계산한 par yield, `zero_cc`: 연속복리 제로, `fwd_1y`: 그 만기에서 시작하는 1년 선도

In [ ]:
with market.active():
    table = curve_table(market["UST"], cfg.conventions["UST"])
(table[["par", "zero_cc", "fwd_1y"]] * 100).round(3)

In [ ]:
ax = (table[["par", "zero_cc", "fwd_1y"]] * 100).set_index(table["years"]).plot(marker="o")
ax.set_xlabel("years")
ax.set_ylabel("%")
ax.set_title(f"UST curve {ASOF}" + (" (synthetic sample)" if SAMPLE else ""))
plt.show()
curve_shape(table).round(1)

## 3. 모양 지표 추이

In [ ]:
history = {}
for d in list(quotes)[-60:]:
    m = build_market(d, quotes[d], cfg)
    with m.active():
        history[d] = curve_shape(curve_table(m["UST"], cfg.conventions["UST"]))
pd.DataFrame(history).T.tail()

## 4. 포지션 key rate DV01

In [ ]:
book = Portfolio.from_file(ROOT / "positions" / "ust_book.yaml").build(market, cfg)
krd = key_rate(book, market, cfg.risk, "UST")["delta"]
print("Σ key rate DV01 =", round(krd["total"].sum(), 2))
ax = krd.drop(columns="total").plot.bar(stacked=True, figsize=(10, 4))
krd["total"].plot(ax=ax, color="k", marker="o", label="total")
ax.set_ylabel("PnL per 1bp (USD)")
ax.legend()
plt.show()

읽는 법: 막대 부호 — 롱은 금리 상승에 손실(음수), 숏은 이익. `total`이 몰린 만기가 실질 듀레이션 위치다. 평행 DV01이 작아도 만기별로 상쇄된 것이면 기울기(2s10s, 5s30s) 위험은 남는다. 이 DV01은 격자 제로금리 기준이다.